In [ ]:
-- 03_Gold_Setup.sql
-- One-time Warehouse setup. Run once, or rerun when procedure definitions change.
-- Creates Gold tables and stored procedures only. Does NOT execute daily loads.

IF OBJECT_ID('dbo.dim_date','U') IS NULL
BEGIN

    CREATE TABLE dbo.dim_date
    (
        DateKey        INT          NOT NULL,
        FullDate       DATE         NOT NULL,
        DayOfMonth     INT          NOT NULL,
        DayName        VARCHAR(10)  NOT NULL,
        DayOfWeek      INT          NOT NULL,
        MonthNumber    INT          NOT NULL,
        MonthName      VARCHAR(10)  NOT NULL,
        QuarterNumber  INT          NOT NULL,
        QuarterName    VARCHAR(2)   NOT NULL,
        YearNumber     INT          NOT NULL,
        IsWeekend      BIT          NOT NULL
    );
END;

GO

IF OBJECT_ID('dbo.dim_passenger', 'U') IS NULL
BEGIN

    CREATE TABLE dbo.dim_passenger
    (
        PassengerKey   BIGINT        NOT NULL,
        PassengerID    VARCHAR(100)  NOT NULL,
        PassengerName  VARCHAR(200)  NULL,
        PassengerPhone VARCHAR(50)   NULL,
        PassengerEmail VARCHAR(320)  NULL,

        EffectiveFrom  DATETIME2(6)  NOT NULL,
        EffectiveTo    DATETIME2(6)  NULL,
        IsCurrent      BIT           NOT NULL
    );

END;

GO

IF OBJECT_ID('dbo.dim_driver', 'U') IS NULL
BEGIN
    CREATE TABLE dbo.dim_driver
    (
        DriverKey       BIGINT        NOT NULL,
        DriverID        VARCHAR(100)  NOT NULL,
        DriverLicense   VARCHAR(100)  NULL,
        DriverName      VARCHAR(200)  NULL,
        DriverPhone     VARCHAR(100)  NULL,

        EffectiveFrom   DATETIME2(6)  NOT NULL,
        EffectiveTo     DATETIME2(6)  NULL,
        IsCurrent       BIT           NOT NULL
    );
END;


GO

IF OBJECT_ID('dbo.dim_city', 'U') IS NULL
BEGIN
    CREATE TABLE dbo.dim_city
    (
        CityKey         BIGINT        NOT NULL,
        CityID          BIGINT        NOT NULL,
        CityName        VARCHAR(200)  NOT NULL,
        Region          VARCHAR(200)  NULL,
        StateName       VARCHAR(200)  NULL,

        EffectiveFrom   DATETIME2(6)  NOT NULL,
        EffectiveTo     DATETIME2(6)  NULL,
        IsCurrent       BIT           NOT NULL
    );
END;

GO

IF OBJECT_ID('dbo.dim_vehicle_type', 'U') IS NULL
BEGIN
    CREATE TABLE dbo.dim_vehicle_type
    (
        VehicleTypeKey  BIGINT         NOT NULL,
        VehicleTypeID   BIGINT         NOT NULL,
        VehicleType     VARCHAR(100)   NOT NULL,
        Description     VARCHAR(500)   NULL,

        BaseRate        DECIMAL(18,2)  NULL,
        PerMile         DECIMAL(18,2)  NULL,
        PerMinute       DECIMAL(18,2)  NULL,

        EffectiveFrom   DATETIME2(6)   NOT NULL,
        EffectiveTo     DATETIME2(6)   NULL,
        IsCurrent       BIT            NOT NULL
    );
END;

GO

IF OBJECT_ID('dbo.dim_vehicle_make', 'U') IS NULL
BEGIN
    CREATE TABLE dbo.dim_vehicle_make
    (
        VehicleMakeID   BIGINT        NOT NULL,
        VehicleMake     VARCHAR(100)  NOT NULL
    );
END;


GO

IF OBJECT_ID('dbo.dim_payment_method', 'U') IS NULL
BEGIN
    CREATE TABLE dbo.dim_payment_method
    (
        PaymentMethodID BIGINT        NOT NULL,
        PaymentMethod   VARCHAR(100)  NOT NULL,
        IsCard          BIT           NULL,
        RequiresAuth    BIT           NULL
    );
END;


GO

IF OBJECT_ID('dbo.dim_ride_status', 'U') IS NULL
BEGIN
    CREATE TABLE dbo.dim_ride_status
    (
        RideStatusID    BIGINT        NOT NULL,
        RideStatus      VARCHAR(100)  NOT NULL,
        IsCompleted     BIT           NULL
    );
END;

GO

IF OBJECT_ID('dbo.dim_cancellation_reason', 'U') IS NULL
BEGIN
    CREATE TABLE dbo.dim_cancellation_reason
    (
        CancellationReasonID BIGINT        NOT NULL,
        CancellationReason   VARCHAR(300)  NOT NULL
    );
END;

GO

CREATE OR ALTER PROCEDURE dbo.usp_load_dim_passenger
AS
BEGIN
    SET NOCOUNT ON;

    -- ========================================================
    -- STEP 1: Synchronize existing passenger versions
    -- ========================================================

    UPDATE target
    SET
        target.PassengerID    = source.passenger_id,
        target.PassengerName  = source.passenger_name,
        target.PassengerPhone = source.passenger_phone,
        target.PassengerEmail = source.passenger_email,
        target.EffectiveFrom  = source.effective_from,
        target.EffectiveTo    = source.effective_to,
        target.IsCurrent      = source.is_current

    FROM dbo.dim_passenger AS target

    INNER JOIN [Uber_Lakehouse].[LH_Silver].[passenger_scd] AS source
        ON target.PassengerKey = source.passenger_key

    WHERE
           ISNULL(target.PassengerID, '')
                <> ISNULL(source.passenger_id, '')

        OR ISNULL(target.PassengerName, '')
                <> ISNULL(source.passenger_name, '')

        OR ISNULL(target.PassengerPhone, '')
                <> ISNULL(source.passenger_phone, '')

        OR ISNULL(target.PassengerEmail, '')
                <> ISNULL(source.passenger_email, '')

        OR target.EffectiveFrom <> source.effective_from

        OR ISNULL(
               target.EffectiveTo,
               CAST('9999-12-31' AS DATETIME2(6))
           )
           <>
           ISNULL(
               source.effective_to,
               CAST('9999-12-31' AS DATETIME2(6))
           )

        OR target.IsCurrent <> source.is_current;


    -- ========================================================
    -- STEP 2: Insert new passenger SCD versions
    -- ========================================================

    INSERT INTO dbo.dim_passenger
    (
        PassengerKey,
        PassengerID,
        PassengerName,
        PassengerPhone,
        PassengerEmail,
        EffectiveFrom,
        EffectiveTo,
        IsCurrent
    )

    SELECT
        source.passenger_key,
        source.passenger_id,
        source.passenger_name,
        source.passenger_phone,
        source.passenger_email,
        source.effective_from,
        source.effective_to,
        source.is_current

    FROM [Uber_Lakehouse].[LH_Silver].[passenger_scd] AS source

    WHERE NOT EXISTS
    (
        SELECT 1
        FROM dbo.dim_passenger AS target
        WHERE target.PassengerKey = source.passenger_key
    );

END;

GO

CREATE OR ALTER PROCEDURE dbo.usp_load_dim_driver
AS
BEGIN
    SET NOCOUNT ON;

    -- ========================================================
    -- STEP 1: Synchronize existing Driver SCD2 versions
    -- ========================================================

    UPDATE target
    SET
        target.DriverID       = source.driver_id,
        target.DriverLicense  = source.driver_license,
        target.DriverName     = source.driver_name,
        target.DriverPhone    = source.driver_phone,
        target.EffectiveFrom  = source.effective_from,
        target.EffectiveTo    = source.effective_to,
        target.IsCurrent      = source.is_current

    FROM dbo.dim_driver AS target

    INNER JOIN [Uber_Lakehouse].[LH_Silver].[driver_scd] AS source
        ON target.DriverKey = source.driver_key

    WHERE
           ISNULL(target.DriverID, '')
                <> ISNULL(source.driver_id, '')

        OR ISNULL(target.DriverLicense, '')
                <> ISNULL(source.driver_license, '')

        OR ISNULL(target.DriverName, '')
                <> ISNULL(source.driver_name, '')

        OR ISNULL(target.DriverPhone, '')
                <> ISNULL(source.driver_phone, '')

        OR target.EffectiveFrom <> source.effective_from

        OR ISNULL(
               target.EffectiveTo,
               CAST('9999-12-31' AS DATETIME2(6))
           )
           <>
           ISNULL(
               source.effective_to,
               CAST('9999-12-31' AS DATETIME2(6))
           )

        OR target.IsCurrent <> source.is_current;


    -- ========================================================
    -- STEP 2: Insert new Driver SCD2 versions
    -- ========================================================

    INSERT INTO dbo.dim_driver
    (
        DriverKey,
        DriverID,
        DriverLicense,
        DriverName,
        DriverPhone,
        EffectiveFrom,
        EffectiveTo,
        IsCurrent
    )

    SELECT
        source.driver_key,
        source.driver_id,
        source.driver_license,
        source.driver_name,
        source.driver_phone,
        source.effective_from,
        source.effective_to,
        source.is_current

    FROM [Uber_Lakehouse].[LH_Silver].[driver_scd] AS source

    WHERE NOT EXISTS
    (
        SELECT 1
        FROM dbo.dim_driver AS target
        WHERE target.DriverKey = source.driver_key
    );

END;

GO

CREATE OR ALTER PROCEDURE dbo.usp_load_dim_city
AS
BEGIN
    SET NOCOUNT ON;

    -- ========================================================
    -- STEP 1: Synchronize existing City SCD2 versions
    -- ========================================================

    UPDATE target
    SET
        target.CityID        = source.city_id,
        target.CityName      = source.city,
        target.Region        = source.region,
        target.StateName     = source.state,
        target.EffectiveFrom = source.effective_from,
        target.EffectiveTo   = source.effective_to,
        target.IsCurrent     = source.is_current

    FROM dbo.dim_city AS target

    INNER JOIN [Uber_Lakehouse].[LH_Silver].[city_scd] AS source
        ON target.CityKey = source.city_key

    WHERE
           target.CityID <> source.city_id

        OR ISNULL(target.CityName, '')
            <> ISNULL(source.city, '')

        OR ISNULL(target.Region, '')
            <> ISNULL(source.region, '')

        OR ISNULL(target.StateName, '')
            <> ISNULL(source.state, '')

        OR target.EffectiveFrom <> source.effective_from

        OR ISNULL(
               target.EffectiveTo,
               CAST('9999-12-31' AS DATETIME2(6))
           )
           <>
           ISNULL(
               source.effective_to,
               CAST('9999-12-31' AS DATETIME2(6))
           )

        OR target.IsCurrent <> source.is_current;


    -- ========================================================
    -- STEP 2: Insert new City SCD2 versions
    -- ========================================================

    INSERT INTO dbo.dim_city
    (
        CityKey,
        CityID,
        CityName,
        Region,
        StateName,
        EffectiveFrom,
        EffectiveTo,
        IsCurrent
    )

    SELECT
        source.city_key,
        source.city_id,
        source.city,
        source.region,
        source.state,
        source.effective_from,
        source.effective_to,
        source.is_current

    FROM [Uber_Lakehouse].[LH_Silver].[city_scd] AS source

    WHERE NOT EXISTS
    (
        SELECT 1
        FROM dbo.dim_city AS target
        WHERE target.CityKey = source.city_key
    );

END;

GO

CREATE OR ALTER PROCEDURE dbo.usp_load_dim_vehicle_type
AS
BEGIN
    SET NOCOUNT ON;

    -- Update existing SCD2 versions
    UPDATE target
    SET
        target.VehicleTypeID = source.vehicle_type_id,
        target.VehicleType   = source.vehicle_type,
        target.Description   = source.description,
        target.BaseRate      = source.base_rate,
        target.PerMile       = source.per_mile,
        target.PerMinute     = source.per_minute,
        target.EffectiveFrom = source.effective_from,
        target.EffectiveTo   = source.effective_to,
        target.IsCurrent     = source.is_current

    FROM dbo.dim_vehicle_type AS target

    INNER JOIN [Uber_Lakehouse].[LH_Silver].[vehicle_type_scd] AS source
        ON target.VehicleTypeKey = source.vehicle_type_key

    WHERE
           target.VehicleTypeID <> source.vehicle_type_id
        OR ISNULL(target.VehicleType, '') <> ISNULL(source.vehicle_type, '')
        OR ISNULL(target.Description, '') <> ISNULL(source.description, '')
        OR ISNULL(target.BaseRate, -1) <> ISNULL(source.base_rate, -1)
        OR ISNULL(target.PerMile, -1) <> ISNULL(source.per_mile, -1)
        OR ISNULL(target.PerMinute, -1) <> ISNULL(source.per_minute, -1)
        OR target.EffectiveFrom <> source.effective_from
        OR ISNULL(
               target.EffectiveTo,
               CAST('9999-12-31' AS DATETIME2(6))
           )
           <>
           ISNULL(
               source.effective_to,
               CAST('9999-12-31' AS DATETIME2(6))
           )
        OR target.IsCurrent <> source.is_current;


    -- Insert new SCD2 versions
    INSERT INTO dbo.dim_vehicle_type
    (
        VehicleTypeKey,
        VehicleTypeID,
        VehicleType,
        Description,
        BaseRate,
        PerMile,
        PerMinute,
        EffectiveFrom,
        EffectiveTo,
        IsCurrent
    )
    SELECT
        source.vehicle_type_key,
        source.vehicle_type_id,
        source.vehicle_type,
        source.description,
        source.base_rate,
        source.per_mile,
        source.per_minute,
        source.effective_from,
        source.effective_to,
        source.is_current

    FROM [Uber_Lakehouse].[LH_Silver].[vehicle_type_scd] AS source

    WHERE NOT EXISTS
    (
        SELECT 1
        FROM dbo.dim_vehicle_type AS target
        WHERE target.VehicleTypeKey = source.vehicle_type_key
    );

END;

GO

CREATE OR ALTER PROCEDURE dbo.usp_load_dim_vehicle_make
AS
BEGIN
    SET NOCOUNT ON;

    UPDATE target
    SET
        target.VehicleMake = source.vehicle_make
    FROM dbo.dim_vehicle_make AS target
    INNER JOIN [Uber_Lakehouse].[LH_Silver].[map_vehicle_makes] AS source
        ON target.VehicleMakeID = source.vehicle_make_id
    WHERE ISNULL(target.VehicleMake, '')
       <> ISNULL(source.vehicle_make, '');

    INSERT INTO dbo.dim_vehicle_make
    (
        VehicleMakeID,
        VehicleMake
    )
    SELECT
        source.vehicle_make_id,
        source.vehicle_make
    FROM [Uber_Lakehouse].[LH_Silver].[map_vehicle_makes] AS source
    WHERE NOT EXISTS
    (
        SELECT 1
        FROM dbo.dim_vehicle_make AS target
        WHERE target.VehicleMakeID = source.vehicle_make_id
    );
END;

GO

CREATE OR ALTER PROCEDURE dbo.usp_load_dim_payment_method
AS
BEGIN
    SET NOCOUNT ON;

    UPDATE target
    SET
        target.PaymentMethod = source.payment_method,
        target.IsCard        = source.is_card,
        target.RequiresAuth  = source.requires_auth
    FROM dbo.dim_payment_method AS target
    INNER JOIN [Uber_Lakehouse].[LH_Silver].[map_payment_methods] AS source
        ON target.PaymentMethodID = source.payment_method_id
    WHERE
           ISNULL(target.PaymentMethod, '')
               <> ISNULL(source.payment_method, '')
        OR ISNULL(target.IsCard, 0)
               <> ISNULL(source.is_card, 0)
        OR ISNULL(target.RequiresAuth, 0)
               <> ISNULL(source.requires_auth, 0);

    INSERT INTO dbo.dim_payment_method
    (
        PaymentMethodID,
        PaymentMethod,
        IsCard,
        RequiresAuth
    )
    SELECT
        source.payment_method_id,
        source.payment_method,
        source.is_card,
        source.requires_auth
    FROM [Uber_Lakehouse].[LH_Silver].[map_payment_methods] AS source
    WHERE NOT EXISTS
    (
        SELECT 1
        FROM dbo.dim_payment_method AS target
        WHERE target.PaymentMethodID = source.payment_method_id
    );
END;

GO

CREATE OR ALTER PROCEDURE dbo.usp_load_dim_ride_status
AS
BEGIN
    SET NOCOUNT ON;

    UPDATE target
    SET
        target.RideStatus  = source.ride_status,
        target.IsCompleted = source.is_completed
    FROM dbo.dim_ride_status AS target
    INNER JOIN [Uber_Lakehouse].[LH_Silver].[map_ride_statuses] AS source
        ON target.RideStatusID = source.ride_status_id
    WHERE
           ISNULL(target.RideStatus, '')
               <> ISNULL(source.ride_status, '')
        OR ISNULL(target.IsCompleted, 0)
               <> ISNULL(source.is_completed, 0);

    INSERT INTO dbo.dim_ride_status
    (
        RideStatusID,
        RideStatus,
        IsCompleted
    )
    SELECT
        source.ride_status_id,
        source.ride_status,
        source.is_completed
    FROM [Uber_Lakehouse].[LH_Silver].[map_ride_statuses] AS source
    WHERE NOT EXISTS
    (
        SELECT 1
        FROM dbo.dim_ride_status AS target
        WHERE target.RideStatusID = source.ride_status_id
    );
END;

GO

CREATE OR ALTER PROCEDURE dbo.usp_load_dim_cancellation_reason
AS
BEGIN
    SET NOCOUNT ON;

    UPDATE target
    SET
        target.CancellationReason = source.cancellation_reason
    FROM dbo.dim_cancellation_reason AS target
    INNER JOIN [Uber_Lakehouse].[LH_Silver].[map_cancellation_reasons] AS source
        ON target.CancellationReasonID = source.cancellation_reason_id
    WHERE ISNULL(target.CancellationReason, '')
       <> ISNULL(source.cancellation_reason, '');

    INSERT INTO dbo.dim_cancellation_reason
    (
        CancellationReasonID,
        CancellationReason
    )
    SELECT
        source.cancellation_reason_id,
        source.cancellation_reason
    FROM [Uber_Lakehouse].[LH_Silver].[map_cancellation_reasons] AS source
    WHERE NOT EXISTS
    (
        SELECT 1
        FROM dbo.dim_cancellation_reason AS target
        WHERE target.CancellationReasonID = source.cancellation_reason_id
    );
END;

GO

IF OBJECT_ID('dbo.fact_rides','U') IS NULL
BEGIN
    CREATE TABLE dbo.fact_rides
    (
        -- ========================================================
        -- BUSINESS IDENTIFIERS
        -- ========================================================

        RideID                  VARCHAR(100)   NOT NULL,
        ConfirmationNumber      VARCHAR(100)   NULL,
        VehicleID               VARCHAR(100)   NULL,


        -- ========================================================
        -- DIMENSION KEYS
        -- ========================================================

        DateKey                 INT            NOT NULL,

        PassengerKey            BIGINT         NOT NULL,
        DriverKey               BIGINT         NOT NULL,

        PickupCityKey           BIGINT         NOT NULL,
        DropoffCityKey          BIGINT         NOT NULL,

        VehicleTypeKey          BIGINT         NOT NULL,

        VehicleMakeID           BIGINT         NULL,
        PaymentMethodID         BIGINT         NULL,
        RideStatusID            BIGINT         NULL,
        CancellationReasonID    BIGINT         NULL,


        -- ========================================================
        -- EVENT TIMESTAMPS
        -- ========================================================

        BookingTimestamp        DATETIME2(6)   NOT NULL,
        PickupTimestamp         DATETIME2(6)   NULL,
        DropoffTimestamp        DATETIME2(6)   NULL,


        -- ========================================================
        -- RIDE MEASURES
        -- ========================================================

        DistanceMiles           DECIMAL(18,2)  NULL,
        DurationMinutes         DECIMAL(18,2)  NULL,

        Rating                  DECIMAL(5,2)   NULL,
        DriverRating            DECIMAL(5,2)   NULL,


        -- ========================================================
        -- FARE MEASURES
        -- ========================================================

        BaseFare                DECIMAL(18,2)  NULL,
        DistanceFare            DECIMAL(18,2)  NULL,
        TimeFare                DECIMAL(18,2)  NULL,

        SurgeMultiplier         DECIMAL(10,2)  NULL,

        Subtotal                DECIMAL(18,2)  NULL,
        TipAmount               DECIMAL(18,2)  NULL,
        TotalFare               DECIMAL(18,2)  NULL
    );
END

GO

CREATE OR ALTER PROCEDURE dbo.usp_load_fact_rides
AS
BEGIN
    SET NOCOUNT ON;

    -- ========================================================
    -- STEP 1: UPDATE EXISTING RIDES THAT HAVE CHANGED
    -- ========================================================

    UPDATE target
    SET
        target.ConfirmationNumber   = source.confirmation_number,
        target.VehicleID            = source.vehicle_id,

        target.DateKey              =
            YEAR(source.booking_timestamp) * 10000
            + MONTH(source.booking_timestamp) * 100
            + DAY(source.booking_timestamp),

        target.PassengerKey         = source.passenger_key,
        target.DriverKey            = source.driver_key,
        target.PickupCityKey        = source.pickup_city_key,
        target.DropoffCityKey       = source.dropoff_city_key,
        target.VehicleTypeKey       = source.vehicle_type_key,

        target.VehicleMakeID        = source.vehicle_make_id,
        target.PaymentMethodID      = source.payment_method_id,
        target.RideStatusID         = source.ride_status_id,
        target.CancellationReasonID = source.cancellation_reason_id,

        target.BookingTimestamp     = source.booking_timestamp,
        target.PickupTimestamp      = source.pickup_timestamp,
        target.DropoffTimestamp     = source.dropoff_timestamp,

        target.DistanceMiles        = source.distance_miles,
        target.DurationMinutes      = source.duration_minutes,

        target.Rating               = source.rating,
        target.DriverRating         = source.driver_rating,

        target.BaseFare             = source.base_fare,
        target.DistanceFare         = source.distance_fare,
        target.TimeFare             = source.time_fare,
        target.SurgeMultiplier      = source.surge_multiplier,
        target.Subtotal             = source.subtotal,
        target.TipAmount            = source.tip_amount,
        target.TotalFare            = source.total_fare

    FROM dbo.fact_rides AS target

    INNER JOIN [Uber_Lakehouse].[LH_Silver].[rides_obt] AS source
        ON target.RideID = source.ride_id

    WHERE
           ISNULL(target.ConfirmationNumber, '')
               <> ISNULL(source.confirmation_number, '')

        OR ISNULL(target.VehicleID, '')
               <> ISNULL(source.vehicle_id, '')

        OR target.PassengerKey <> source.passenger_key
        OR target.DriverKey <> source.driver_key
        OR target.PickupCityKey <> source.pickup_city_key
        OR target.DropoffCityKey <> source.dropoff_city_key
        OR target.VehicleTypeKey <> source.vehicle_type_key

        OR ISNULL(target.VehicleMakeID, -1)
               <> ISNULL(source.vehicle_make_id, -1)

        OR ISNULL(target.PaymentMethodID, -1)
               <> ISNULL(source.payment_method_id, -1)

        OR ISNULL(target.RideStatusID, -1)
               <> ISNULL(source.ride_status_id, -1)

        OR ISNULL(target.CancellationReasonID, -1)
               <> ISNULL(source.cancellation_reason_id, -1)

        OR target.BookingTimestamp <> source.booking_timestamp

        OR ISNULL(target.PickupTimestamp, CAST('1900-01-01' AS DATETIME2(6)))
               <> ISNULL(source.pickup_timestamp, CAST('1900-01-01' AS DATETIME2(6)))

        OR ISNULL(target.DropoffTimestamp, CAST('1900-01-01' AS DATETIME2(6)))
               <> ISNULL(source.dropoff_timestamp, CAST('1900-01-01' AS DATETIME2(6)))

        OR ISNULL(target.DistanceMiles, -1)
               <> ISNULL(source.distance_miles, -1)

        OR ISNULL(target.DurationMinutes, -1)
               <> ISNULL(source.duration_minutes, -1)

        OR ISNULL(target.Rating, -1)
               <> ISNULL(source.rating, -1)

        OR ISNULL(target.DriverRating, -1)
               <> ISNULL(source.driver_rating, -1)

        OR ISNULL(target.BaseFare, -1)
               <> ISNULL(source.base_fare, -1)

        OR ISNULL(target.DistanceFare, -1)
               <> ISNULL(source.distance_fare, -1)

        OR ISNULL(target.TimeFare, -1)
               <> ISNULL(source.time_fare, -1)

        OR ISNULL(target.SurgeMultiplier, -1)
               <> ISNULL(source.surge_multiplier, -1)

        OR ISNULL(target.Subtotal, -1)
               <> ISNULL(source.subtotal, -1)

        OR ISNULL(target.TipAmount, -1)
               <> ISNULL(source.tip_amount, -1)

        OR ISNULL(target.TotalFare, -1)
               <> ISNULL(source.total_fare, -1);


    -- ========================================================
    -- STEP 2: INSERT NEW RIDES
    -- ========================================================

    INSERT INTO dbo.fact_rides
    (
        RideID,
        ConfirmationNumber,
        VehicleID,

        DateKey,

        PassengerKey,
        DriverKey,
        PickupCityKey,
        DropoffCityKey,
        VehicleTypeKey,

        VehicleMakeID,
        PaymentMethodID,
        RideStatusID,
        CancellationReasonID,

        BookingTimestamp,
        PickupTimestamp,
        DropoffTimestamp,

        DistanceMiles,
        DurationMinutes,

        Rating,
        DriverRating,

        BaseFare,
        DistanceFare,
        TimeFare,
        SurgeMultiplier,
        Subtotal,
        TipAmount,
        TotalFare
    )

    SELECT
        source.ride_id,
        source.confirmation_number,
        source.vehicle_id,

        YEAR(source.booking_timestamp) * 10000
        + MONTH(source.booking_timestamp) * 100
        + DAY(source.booking_timestamp),

        source.passenger_key,
        source.driver_key,
        source.pickup_city_key,
        source.dropoff_city_key,
        source.vehicle_type_key,

        source.vehicle_make_id,
        source.payment_method_id,
        source.ride_status_id,
        source.cancellation_reason_id,

        source.booking_timestamp,
        source.pickup_timestamp,
        source.dropoff_timestamp,

        source.distance_miles,
        source.duration_minutes,

        source.rating,
        source.driver_rating,

        source.base_fare,
        source.distance_fare,
        source.time_fare,
        source.surge_multiplier,
        source.subtotal,
        source.tip_amount,
        source.total_fare

    FROM [Uber_Lakehouse].[LH_Silver].[rides_obt] AS source

    WHERE NOT EXISTS
    (
        SELECT 1
        FROM dbo.fact_rides AS target
        WHERE target.RideID = source.ride_id
    );

END;

GO
CREATE OR ALTER PROCEDURE dbo.usp_load_dim_date
AS
BEGIN
 SET NOCOUNT ON;
 DECLARE @MinDate DATE=(SELECT MIN(CAST(booking_timestamp AS DATE)) FROM [Uber_Lakehouse].[LH_Silver].[rides_obt]);
 DECLARE @MaxDate DATE=(SELECT MAX(CAST(booking_timestamp AS DATE)) FROM [Uber_Lakehouse].[LH_Silver].[rides_obt]);
 IF @MinDate IS NULL RETURN;
 ;WITH n AS (SELECT TOP (DATEDIFF(DAY,@MinDate,@MaxDate)+1) ROW_NUMBER() OVER(ORDER BY (SELECT NULL))-1 n FROM sys.all_columns a CROSS JOIN sys.all_columns b),
 d AS (SELECT DATEADD(DAY,n,@MinDate) FullDate FROM n)
 INSERT INTO dbo.dim_date(DateKey,FullDate,DayOfMonth,DayName,DayOfWeek,MonthNumber,MonthName,QuarterNumber,QuarterName,YearNumber,IsWeekend)
 SELECT YEAR(FullDate)*10000+MONTH(FullDate)*100+DAY(FullDate),FullDate,DAY(FullDate),DATENAME(WEEKDAY,FullDate),DATEPART(WEEKDAY,FullDate),MONTH(FullDate),DATENAME(MONTH,FullDate),DATEPART(QUARTER,FullDate),CONCAT('Q',DATEPART(QUARTER,FullDate)),YEAR(FullDate),CASE WHEN DATENAME(WEEKDAY,FullDate) IN ('Saturday','Sunday') THEN 1 ELSE 0 END
 FROM d WHERE NOT EXISTS(SELECT 1 FROM dbo.dim_date x WHERE x.FullDate=d.FullDate);
END;